# Stage 11 — Option buying live screener

New: `src/option_buying_screener.py` - parallel to `debit_spread_screener.py`, not a modification. Checks for a currently-OPEN naked-buying trade (a signal that fired recently and hasn't yet resolved via stop/trail-stop/time-exit) — the actionable entry itself, with live stop level, whether trailing has activated, and candles left before the time-exit forces a close.

**4H trend hard filter is active** (`apply_4h_trend_filter()`, baked into `option_buying_scale_pipeline.process_instrument_incremental()`, which this notebook calls) — an entry only shows up here if the 1H EMA/ADX signal agrees with the symbol's own 4H Supertrend direction at that moment. Tested finding: aligned entries won 53.3% vs 47.2% against, median R +0.08 vs -0.46 (12,608 real trades, see README Stage 11). This roughly halves how often you'll see an active trade here vs before the filter - that's the filter working, not a bug.

Same two-speed discipline as the debit spread screener: deep full-universe refresh (notebook 15, ~monthly) keeps the ranking honest; THIS notebook does a quick, short-lookback refresh of only the shortlist, right before checking - never point this at the full universe.

**Regression fixed:** this refresh cell used to call `process_instrument()`, which does a full replace of the raw 1H file - every run was silently overwriting notebook 15's deep 2023-2026 history for whatever's in the shortlist with just a 120-day window (same class of bug already found once in the pledged-securities pipeline). Confirmed directly: ADANIENSOL/KPITTECH/BSE collapsed from 6,194 rows to 570 rows this way, producing a corrupted near-empty trades file for ADANIENSOL. Now uses `process_instrument_incremental()`, which merges the quick catch-up window onto existing history instead of replacing it, and refuses to save if the merge would shrink history. If you ran this notebook before the fix, re-run notebook 15 (full-universe deep refresh) once to restore any shortlisted symbols that got shrunk.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

ranking = pd.read_excel("../exports/RANKING_option_buying.xlsx")

# Keep only outlier-reliance-clean stocks, take the top 20
clean_candidates = ranking[
    ranking["outlier_reliance_flag"].isna() | (ranking["outlier_reliance_flag"] == "")
]
top_20 = clean_candidates.head(20)

# Always include NIFTY and BANKNIFTY, same flag check
must_include = ranking[ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["outlier_reliance_flag"].isna() | (must_include["outlier_reliance_flag"] == "")]

final_candidates = pd.concat([top_20, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates[["symbol", "option_buying_score", "win_rate_pct", "avg_r_achieved", "median_r_achieved"]]

,symbol,option_buying_score,win_rate_pct,avg_r_achieved,median_r_achieved
0,PFC,92.7,69.2,0.72,0.62
1,RVNL,92.0,73.8,0.79,0.68
2,GMRAIRPORT,91.9,73.5,0.52,0.39
3,ADANIENSOL,91.2,63.6,0.49,0.40
4,COFORGE,91.2,64.3,0.68,0.42
5,KPITTECH,90.9,75.7,0.58,0.62
6,GODREJPROP,89.4,71.9,0.48,0.32
7,BDL,88.9,68.4,0.56,0.30
8,HINDUNILVR,87.8,64.9,0.86,0.77
9,SUZLON,87.7,59.5,0.72,0.48


In [2]:
from datetime import datetime
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.option_buying_scale_pipeline import process_instrument_incremental

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

# REGRESSION FIX: this used to call process_instrument() with a short
# 120-day from_date, which does a full REPLACE of the raw 1H file - that
# silently destroyed notebook 15's deep 2023-2026 history for every
# shortlisted symbol on every screener run (confirmed: ADANIENSOL/
# KPITTECH/BSE collapsed from 6,194 rows to 570 rows this way, producing
# a corrupted near-empty trades file). Now uses process_instrument_incremental()
# instead - same quick 120-day catch-up fetch, but MERGED onto existing
# saved history rather than replacing it, with a guard that refuses to
# save if history would shrink. No from_date needed here; the incremental
# function derives its own fetch window from what's already saved.

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_instrument_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-04-09 to 2026-08-07...
Saved 6197 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/PFC_60minute.csv
Saved 26 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buying/PFC_trades.csv
[1/21] PFC — refreshed
Fetching 2026-04-09 to 2026-08-07...
Saved 6197 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/RVNL_60minute.csv
Saved 42 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buying/RVNL_trades.csv
[2/21] RVNL — refreshed
Fetching 2026-04-09 to 2026-08-07...
Saved 6197 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/GMRAIRPORT_60minute.csv
Saved 34 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_har

In [3]:
from src.option_buying_screener import screen_shortlist

active = screen_shortlist(symbols_to_refresh)
print(f"{len(active)} of {len(symbols_to_refresh)} shortlisted instruments have an active trade right now")
active

2 of 21 shortlisted instruments have an active trade right now


,symbol,direction,entry_date,entry_price,initial_stop_price,current_stop_price,trailing_activated,candles_held,candles_left_before_time_exit,current_pct_move,current_r
0,GVT&D,DOWN,2026-08-07 09:15:00,4313.2,4425.94,4425.94,False,3,11,-0.20,-0.08
1,DMART,DOWN,2026-08-06 14:15:00,3945.1,3983.05,3932.95,True,4,10,1.07,1.11
